Financial budget 2026-27

In [12]:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
loader=DirectoryLoader("../data/pdf",
                    glob="**/*.pdf",
                    show_progress=False,
                    loader_cls=PyPDFLoader
                    
                    )
pdf_doc=loader.load()
print(pdf_doc)

[Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}, page_content='GOVERNMENT OF INDIA\nBUDGET 2026-2027\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2026'), Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 1, 'page_label': '2'}, page_content=''), Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages'

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_text(document,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                chunk_overlap=chunk_overlap,
                                                length_function=len,
                                                separators=["\n\n","\n"," ",""])
    
    split_docs = text_splitter.split_documents(document)
    print(f"Total chunks created: {len(split_docs)}")
    
    if split_docs:
        print("\nSample chunk:")
        print(f"content: {split_docs[0].page_content[:500]}...")# Print the first 500 characters of the first chunk
        print(f"metadata: {split_docs[0].metadata}")
    return split_docs

In [14]:
chunks=split_text(pdf_doc)
chunks

Total chunks created: 149

Sample chunk:
content: GOVERNMENT OF INDIA
BUDGET 2026-2027
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2026...
metadata: {'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}, page_content='GOVERNMENT OF INDIA\nBUDGET 2026-2027\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2026'),
 Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 2, 'page_label': '3'}, page_content='CONTENTS \n \nPage No. \nIntroduction 1 \n                                                          PART - A                                          \nYuva Shakti and 3 kartavya 2 \nReform Express 3 \nFirst kartavya: to accelerate and sustain economic growth  3 

### embedding of the chunks and storing in vectordb

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Tuple,Any
from sklearn.metrics.pairwise import cosine_similarity


In [16]:
class EmbeddingManager:
    def __init__(self,model_name: str ="all-MiniLM-L6-V2"):
        '''
        initialize the embedding manager
        
        args:
            model_name: hugging face model name for sentence transformer
        '''
        self.model_name=model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        """load the sentence transformer model"""
        try:
            print(f"loading model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print("model loaded successfully . embedding dimentions: ",self.model.get_sentence_embedding_dimension())
        except Exception as e:  
            print(f"error loading model: {e}")
            raise
        
    def generate_embedding(self,text:List[str])->np.ndarray:
        '''
        generate embeddings for a list of texts
        
        args:
            text: list of text strings to embed
            
        returns:
            np.ndarray: array of embeddings with shape (len(text), embedding_dimension)
        '''
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        print(f"generating embeddings for {len(text)} texts")
        embeddings=self.model.encode(text,show_progress_bar=True)
        print(f"generated embeddings for {embeddings.shape}")
        return embeddings

#initiale embedding mananger
embedding_manager=EmbeddingManager()
embedding_manager
            

loading model: all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 578.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully . embedding dimentions:  384


### vector store

In [21]:
import os
class VetorStore:
    '''
    A simple vector store using ChromaDB to store and retrieve document embeddings.
    '''
    def __init__(self,collection_name: str="budget",persist_directory: str="../data/chroma_db"):
        '''
        initialize the vector store
        
        args:
            collection_name: name of the collection to store embeddings
            persist_directory: directory to persist the ChromaDB database
        '''
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
        
    def _initialize_store(self):
        '''initialize the ChromaDB client and collection'''
        
        try:
            #create persistent chroma_db client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            
            #get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing document embeddings related to financial budget analysis"}
            )
            print(f'vector store initialized . collection : {self.collection_name}')
            print(f'existing documents in collection: {self.collection.count()}')
        except Exception as e:
            print(f"error initializing vector store: {e}")
            raise
    
    def add_documents(self,document: List[Any],embedding=np.ndarray):
        ''' 
        add documents and their embeddings to the vector store
        
        args:
            document : list of langchain documents (budget document)
            embeddings : corresponding embeddings to the document
        '''
        if len(document)!=len(embedding):
            raise ValueError("Number of documents and embeddings must match")
        print(f"adding {len(document)} documents")
        
        #prepare data for chromadb
        ids=[]
        metadata=[]
        document_texts=[]
        embeddings_list=[]
        
        for i,(doc,embeddin) in enumerate(zip(document,embedding)):
            #generate unique id
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #prepare metadata
            meta = dict(doc.metadata)
            meta['doc_index']=i
            meta['content_length']=len(doc.page_content)
            metadata.append(meta)
            
            #document text
            document_texts.append(doc.page_content)
            
            #embedding
            embeddings_list.append(embeddin.tolist())
            
        #add to collection
        try:
            self.collection.add(
                ids=ids,
                documents=document_texts,
                embeddings=embeddings_list,
                metadatas=metadata
            )
            print(f"successfully added {len(document)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise
            
vector_store=VetorStore()
vector_store
            
    

vector store initialized . collection : budget
existing documents in collection: 447


In [22]:
chunks

[Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}, page_content='GOVERNMENT OF INDIA\nBUDGET 2026-2027\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2026'),
 Document(metadata={'producer': 'doPDF Ver 8.4 Build 935', 'creator': 'Adobe Acrobat (64-bit) 25.1.20997', 'creationdate': '2026-02-01T05:37:46+05:30', 'moddate': '2026-02-01T05:39:37+05:30', 'title': '', 'source': '../data/pdf/budget_speech.pdf', 'total_pages': 65, 'page': 2, 'page_label': '3'}, page_content='CONTENTS \n \nPage No. \nIntroduction 1 \n                                                          PART - A                                          \nYuva Shakti and 3 kartavya 2 \nReform Express 3 \nFirst kartavya: to accelerate and sustain economic growth  3 

In [23]:
texts=[doc.page_content for doc in chunks]
texts

['GOVERNMENT OF INDIA\nBUDGET 2026-2027\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2026',
 'CONTENTS \n \nPage No. \nIntroduction 1 \n                                                          PART - A                                          \nYuva Shakti and 3 kartavya 2 \nReform Express 3 \nFirst kartavya: to accelerate and sustain economic growth  3 \nSecond kartavya: fulfil aspirations and build capacity 10 \nThird kartavya: Sabka Sath, Sabka Vikas 14  \n16th Finance Commission 18 \nFiscal Consolidation 18 \n \nPART – B \nDirect taxes 20 \nIndirect Taxes  26 \n \nAnnexure to Part-A 32 \nAnnexure to Part-B \nAmendments relating to Direct Taxes 33 \nAmendments relating to Indirect Taxes 50',
 'Budget 2026-2027 \n \nSpeech of \nNirmala Sitharaman \nMinister of Finance \nFebruary 1, 2026 \n \nHon’ble Speaker, \nOn the sacred occasion of Magha Purnima and the birth \nanniversary of Guru Ravidas, I present the Budget for the year 2026-2027. \nIntroduction  \n1. Si

In [24]:
##generate embeddings for the document chunks
embeddings=embedding_manager.generate_embedding(texts)

##adding it to vector store
vector_store.add_documents(chunks,embeddings)


generating embeddings for 149 texts


Batches: 100%|██████████| 5/5 [00:08<00:00,  1.69s/it]


generated embeddings for (149, 384)
adding 149 documents
successfully added 149 documents to vector store
total documents in collection: 596


In [25]:
print(len(chunks), embeddings.shape)

149 (149, 384)


### retriever pipeline from vector store

In [29]:
class RAGRetriever:
    '''handles query-based retrieval from vector store'''
    def __init__(self,embedding_manager: EmbeddingManager,vector_store: VetorStore):
        '''
        initialize the retriever
        
        args:
            embedding_manager : instance of EmbeddingManager to generate query embeddings
            vector_store : instance of VetorStore to retrieve relevant documents
        '''
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store
        
    def retrieve(self,query: str,top_k: int=5,score_threshold: float=0.0) -> list(dict[str,Any]):
        '''
        retrieve relevant documents for a given query
        
        args:
            query: user query string
            top_k: number of top relevant documents to return
            score_threshold: minimum cosine similarity score to consider a document relevant
        '''
        print(f"retrieving document for query: {query}")
        print(f"retrieving top {top_k} documents with threshold: {score_threshold}")
        
        #generate embedding for the query
        query_embedding=self.embedding_manager.generate_embedding([query])[0]
        
        #search in vector store
        
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            #process results
            retrieved_docs=[]
            
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                
                for i,(document,metadata,distance,doc_id) in enumerate(zip(documents,metadatas,distances,ids)):
                    #converting distance to similarity
                    similarity_score=1-distance
                    #if similarity_score>=score_threshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'score': similarity_score,
                        'distance': distance,
                        'rank': i+1
                    })
                print(f"retrieved {len(retrieved_docs)} relevant documents(after filtering)")
            else:
                print("no documents retrieved from vector store")
            return retrieved_docs 
              
        except Exception as e:
            print(f"error retrieving documents: {e}")
            return []
            
ragRetriever=RAGRetriever(embedding_manager,vector_store)             
        

In [30]:
ragRetriever.retrieve(query='what are the budget given to different departments?')

retrieving document for query: what are the budget given to different departments?
retrieving top 5 documents with threshold: 0.0
generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:15<00:00, 15.67s/it]


generated embeddings for (1, 384)
retrieved 5 relevant documents(after filtering)


[{'id': 'doc_ae51e6ec_50',
  'content': '19  \n \nestimated at par with BE of 2025 -26 at 4.4 percent of GDP. In line with \nthe new fiscal prudence path of debt consolidation, the fiscal deficit in BE \n2026-27 is estimated to be 4.3 percent of GDP.  \nRevised Estimates 2025-26 \n95. The Revised Estimates of the non -debt receipts  \nare ₹34 lakh crore   of which the Centre’s net tax receipts  \nare ₹26.7 lakh crore. The Revised Estimate of the total expenditure is \n₹49.6 lakh crore, of which the capital expenditure is about  \n₹11 lakh crore.  \nBudget Estimates 2026-27 \n96. Coming to 2026 -27, the non -debt receipts and the  \ntotal expenditure are estimated as ₹36.5 lakh crore  \nand ₹53.5 lakh crore respectively. The Centre’s net tax receipts are \nestimated at ₹28.7 lakh crore. \n97. To finance the fiscal deficit, the net market borrowings from dated \nsecurities are estimated at ₹11.7 lakh crore. The balance financing is \nexpected to come from small savings and other sources.

### Integration llm output with vectordb context pipeline

In [43]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#initializing the llm 
groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,temperature=0.1,model_name="llama-3.3-70b-versatile",max_tokens=2048)

#simple rag function: retrieve context and generate answer

def rag_simple(query,retriever,llm,top_k=3):
    #retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        print("no relevant context found")
        
    #generate the answer using llm
    prompt=f"""answer the question based on the following context concisely.
    
    context:{context}
    question:{query}
    answer:
    """
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content




In [44]:
answer=rag_simple("Hi-Tech Tool Rooms?",ragRetriever,llm)
answer

retrieving document for query: Hi-Tech Tool Rooms?
retrieving top 3 documents with threshold: 0.0
generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


generated embeddings for (1, 384)
retrieved 3 relevant documents(after filtering)


'Hi-Tech Tool Rooms will be established by CPSEs at 2 locations as digitally enabled automated service bureaus to design, test, and manufacture high-precision components at scale and lower cost.'

In [38]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])
models = client.models.list()

for m in models.data:
    print(m.id)


meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english
moonshotai/kimi-k2-instruct
openai/gpt-oss-120b
qwen/qwen3-32b
whisper-large-v3-turbo
meta-llama/llama-4-scout-17b-16e-instruct
groq/compound-mini
llama-3.3-70b-versatile
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
llama-3.1-8b-instant
groq/compound
whisper-large-v3
moonshotai/kimi-k2-instruct-0905
